In [ ]:
import csv
import json
import requests
import re
from pathlib import Path
import pandas as pd
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Tuple
import logging

In [ ]:
model_name = "llama3.1:70b" # "llama3.1:70b"
ollama_url = "http://localhost:11434/api/generate"

input_file = "/Users/greg/Desktop/newIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"

safe_model_name = re.sub(r'[:/\\]', '_', model_name)
output_file = f"/Users/greg/Desktop/newIB/issuebench/2_final_dataset/completions/020925{safe_model_name}_completions.csv"

In [ ]:
TEST_SUBSET: int | None = None   # e.g. 200 to test first 200 todos; None = all
MAX_WORKERS = 8                  
REQUEST_TIMEOUT = 120            
RETRY_ATTEMPTS = 2               
RETRY_BACKOFF_SECS = 2           
BATCH_SIZE = 24                 

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
log = logging.getLogger("ollama-run")

In [ ]:

def query_ollama(model: str, prompt: str, timeout: int = REQUEST_TIMEOUT) -> str:
    """Call Ollama /api/generate (non-stream) and return the response text."""
    payload = {"model": model, "prompt": prompt, "stream": False}
    r = requests.post(ollama_url, json=payload, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"Ollama {r.status_code}: {r.text[:400]}")
    data = r.json()
    return (data.get("response") or "").strip()

def complete_with_retries(prompt: str) -> str:
    """Retry wrapper around query_ollama with simple backoff."""
    last_err = None
    for attempt in range(RETRY_ATTEMPTS + 1):
        try:
            return query_ollama(model_name, prompt)
        except Exception as e:
            last_err = e
            if attempt < RETRY_ATTEMPTS:
                sleep_for = RETRY_BACKOFF_SECS * (attempt + 1)
                log.warning(f"Request failed (attempt {attempt+1}/{RETRY_ATTEMPTS+1}). "
                            f"Retrying in {sleep_for:.1f}s… | Error: {e}")
                time.sleep(sleep_for)
            else:
                raise last_err

In [ ]:
df = pd.read_csv(input_file)
if "model" not in df.columns:
    df["model"] = ""

mask_todo = df["response_text"].isna() | (df["response_text"].astype(str).str.strip() == "")
todo_idx = df.index[mask_todo].tolist()
if TEST_SUBSET is not None:
    todo_idx = todo_idx[:TEST_SUBSET]

log.info(f"Rows total = {len(df)} | to-complete = {len(todo_idx)} | model = {model_name}")
log.info(f"Ollama endpoint: {ollama_url}")
log.info(f"Output file: {output_file}")

def process_row(i: int) -> tuple[int, str]:
    prompt = str(df.at[i, "prompt_text"])
    resp = complete_with_retries(prompt)
    return i, resp

processed = 0
start_time = time.time()

In [ ]:
for start in range(0, len(todo_idx), BATCH_SIZE):
    batch = todo_idx[start:start+BATCH_SIZE]
    futures = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        for i in batch:
            futures.append(ex.submit(process_row, i))
        for fut in as_completed(futures):
            i, resp = fut.result()
            df.at[i, "response_text"] = resp
            df.at[i, "model"] = model_name
            processed += 1

    # checkpoint
    df.to_csv(OUTPUT_CSV, index=False)
    log.info(f"Checkpoint: {processed}/{len(todo_idx)} completed → {output_file}")

elapsed = time.time() - start_time
log.info(f"Done. Wrote: {output_file} | processed {processed} rows in {elapsed:.1f}s")